<a href="https://colab.research.google.com/github/Mromererg/vindr-spineXR-detection/blob/main/03_yolo_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLO Tam Eğitim

Bu notebook, W&B Sweep'ten elde edilen en iyi hiperparametrelerle YOLO modelinin tam eğitimini yapar.

**Sweep Sonuçları (expert-sweep-38 parametreleri):**
- lr0: 0.002350804
- lrf: 0.049305842
- momentum: 0.960881751
- weight_decay: 4.62e-05
- batch: 16
- mosaic: 0, mixup: 0.2

**Eğitilen Modeller:**
| Model | mAP@0.5 | mAP@0.5:0.95 | Best Epoch |
|-------|---------|--------------|------------|
| YOLO11l | 0.450 | 0.218 | 29 |
| YOLO26l | 0.452 | 0.227 | 33 |

## 1. Kurulum ve Hazırlık

In [ ]:
!pip install ultralytics wandb -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json

DATASET_DIR = '/content/spineXR_yolo_dataset_guncel'
BASE = '/content/drive/MyDrive/dataset/abnormal'
TRAIN_JSON = f'{BASE}/splits/original/train_coco_train.json'
VAL_JSON   = f'{BASE}/splits/original/train_coco_val.json'

os.makedirs(f'{DATASET_DIR}/images/train', exist_ok=True)
os.makedirs(f'{DATASET_DIR}/images/val',   exist_ok=True)
os.makedirs(f'{DATASET_DIR}/labels/train', exist_ok=True)
os.makedirs(f'{DATASET_DIR}/labels/val',   exist_ok=True)

with open(TRAIN_JSON) as f:
    train_data = json.load(f)
with open(VAL_JSON) as f:
    val_data = json.load(f)

for img in train_data['images']:
    src = f"{BASE}/train_pngs/{img['file_name']}"
    dst = f"{DATASET_DIR}/images/train/{img['file_name']}"
    if not os.path.exists(dst): os.symlink(src, dst)

for img in val_data['images']:
    src = f"{BASE}/train_pngs/{img['file_name']}"
    dst = f"{DATASET_DIR}/images/val/{img['file_name']}"
    if not os.path.exists(dst): os.symlink(src, dst)

for f in os.listdir(f'{BASE}/yolo_labels/train'):
    src = f'{BASE}/yolo_labels/train/{f}'
    dst = f'{DATASET_DIR}/labels/train/{f}'
    if not os.path.exists(dst): os.symlink(src, dst)

for f in os.listdir(f'{BASE}/yolo_labels/val'):
    src = f'{BASE}/yolo_labels/val/{f}'
    dst = f'{DATASET_DIR}/labels/val/{f}'
    if not os.path.exists(dst): os.symlink(src, dst)

yaml_content = f"""path: {DATASET_DIR}
train: images/train
val: images/val
nc: 7
names:
  - Osteophytes
  - Spondylolysthesis
  - Disc space narrowing
  - Other lesions
  - Surgical implant
  - Foraminal stenosis
  - Vertebral collapse
"""
with open(f'{DATASET_DIR}/dataset.yaml', 'w') as f:
    f.write(yaml_content)

print('Dataset hazır!')
print('Train images:', len(os.listdir(f'{DATASET_DIR}/images/train')))
print('Val images:',   len(os.listdir(f'{DATASET_DIR}/images/val')))

In [ ]:
import wandb
wandb.login()  # API key girilecek

## 2. YOLO11l Tam Eğitim

expert-sweep-38 parametreleriyle YOLO11l eğitimi.

In [ ]:
from ultralytics import YOLO
import shutil

YAML_PATH = '/content/spineXR_yolo_dataset_guncel/dataset.yaml'

wandb.init(project='spineXR-yolo', name='full-train-yolo11l-expert38')

model = YOLO('yolo11l.pt')
model.train(
    data=YAML_PATH,
    epochs=50,
    imgsz=640,
    batch=16,
    lr0=0.002350804434896538,
    lrf=0.049305842400680845,
    momentum=0.9608817516521524,
    weight_decay=4.623462715781378e-05,
    optimizer='SGD',
    hsv_h=0.05156094152328464,
    hsv_s=0.40985658118474955,
    hsv_v=0.019199456906849155,
    mixup=0.2,
    scale=0.30319738097024995,
    fliplr=0.0,
    flipud=0.0,
    mosaic=0.0,
    degrees=3.3039596255597914,
    translate=0.12046396298015004,
    close_mosaic=10,
    patience=15,
    save=True,
    plots=True,
    verbose=True,
    project='spineXR-yolo',
    name='full-train-yolo11l-expert38',
    exist_ok=True,
)

# Drive'a kaydet
os.makedirs('/content/drive/MyDrive/spine_results/yolo11l_expert38', exist_ok=True)
shutil.copytree(
    '/content/runs/detect/spineXR-yolo/full-train-yolo11l-expert38',
    '/content/drive/MyDrive/spine_results/yolo11l_expert38',
    dirs_exist_ok=True
)
print('YOLO11l kaydedildi!')
wandb.finish()

## 3. YOLO26l Tam Eğitim

expert-sweep-38 parametreleriyle YOLO26l eğitimi.

In [ ]:
wandb.init(project='spineXR-yolo', name='full-train-yolo26l-expert38')

model = YOLO('yolo26l.pt')
model.train(
    data=YAML_PATH,
    epochs=50,
    imgsz=640,
    batch=16,
    lr0=0.002350804434896538,
    lrf=0.049305842400680845,
    momentum=0.9608817516521524,
    weight_decay=4.623462715781378e-05,
    optimizer='SGD',
    hsv_h=0.05156094152328464,
    hsv_s=0.40985658118474955,
    hsv_v=0.019199456906849155,
    mixup=0.2,
    scale=0.30319738097024995,
    fliplr=0.0,
    flipud=0.0,
    mosaic=0.0,
    degrees=3.3039596255597914,
    translate=0.12046396298015004,
    close_mosaic=10,
    patience=15,
    save=True,
    plots=True,
    verbose=True,
    project='spineXR-yolo',
    name='full-train-yolo26l-expert38',
    exist_ok=True,
)

# Drive'a kaydet
os.makedirs('/content/drive/MyDrive/spine_results/yolo26l_expert38', exist_ok=True)
shutil.copytree(
    '/content/runs/detect/spineXR-yolo/full-train-yolo26l-expert38',
    '/content/drive/MyDrive/spine_results/yolo26l_expert38',
    dirs_exist_ok=True
)
print('YOLO26l kaydedildi!')
wandb.finish()

## 4. Sonuçları Karşılaştır

In [ ]:
# YOLO11l sonuçları
results = {
    'YOLO11l': {
        'mAP50': 0.450,
        'mAP50_95': 0.218,
        'best_epoch': 29,
        'classes': {
            'Osteophytes': 0.410,
            'Spondylolysthesis': 0.482,
            'Disc space narrowing': 0.296,
            'Other lesions': 0.073,
            'Surgical implant': 0.706,
            'Foraminal stenosis': 0.519,
            'Vertebral collapse': 0.664,
        }
    },
    'YOLO26l': {
        'mAP50': 0.452,
        'mAP50_95': 0.227,
        'best_epoch': 33,
        'classes': {
            'Osteophytes': 0.418,
            'Spondylolysthesis': 0.385,
            'Disc space narrowing': 0.287,
            'Other lesions': 0.059,
            'Surgical implant': 0.711,
            'Foraminal stenosis': 0.554,
            'Vertebral collapse': 0.749,
        }
    }
}

print(f"{'Sınıf':<25} {'YOLO11l':>10} {'YOLO26l':>10}")
print('-' * 47)
for cls in results['YOLO11l']['classes']:
    v11 = results['YOLO11l']['classes'][cls]
    v26 = results['YOLO26l']['classes'][cls]
    print(f"{cls:<25} {v11:>10.3f} {v26:>10.3f}")
print('-' * 47)
print(f"{'Genel mAP@0.5':<25} {results['YOLO11l']['mAP50']:>10.3f} {results['YOLO26l']['mAP50']:>10.3f}")
print(f"{'mAP@0.5:0.95':<25} {results['YOLO11l']['mAP50_95']:>10.3f} {results['YOLO26l']['mAP50_95']:>10.3f}")